In [4]:
from pathlib import Path
import sqlite3
import pandas as pd
from prefect import flow, task
from prefect.logging import get_run_logger

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

PROJECT_ROOT = BASE_DIR.parent
DB_PATH = PROJECT_ROOT / "data" / "shopdata.db"

OUTPUT_DB_PATH = BASE_DIR / "analytics.db"
CLEAN_CUSTOMERS_CSV = BASE_DIR / "clean_customers.csv"
CLEAN_ORDERS_CSV = BASE_DIR / "clean_orders.csv"


@task(retries=2, retry_delay_seconds=5)
def extract(db_path: str) -> dict:
    logger = get_run_logger()
    logger.info(f"Starting extract from {db_path}")

    try:
        tables = ["vw_raw_customers", "vw_raw_orders", "vw_exchange_rates"]
        extracted_data = {}

        with sqlite3.connect(db_path) as conn:
            for table in tables:
                query = f"SELECT * FROM {table}"
                df = pd.read_sql_query(query, conn)
                extracted_data[table] = df
                logger.info(f"Loaded {len(df)} rows from {table}")

        logger.info("Extract completed successfully")
        return extracted_data

    except Exception:
        logger.exception("Extract failed")
        raise


@flow(name="etl-pipeline")
def etl_pipeline(db_path: str = str(DB_PATH)) -> None:
    logger = get_run_logger()
    logger.info("ETL pipeline started")

    try:
        data = extract(db_path)

    except Exception:
        logger.exception("ETL pipeline failed")
        raise


if __name__ == "__main__":
    etl_pipeline()


08:25:41.988 | INFO    | Flow run 'fierce-jackrabbit' - Beginning flow run 'fierce-jackrabbit' for flow 'etl-pipeline'

08:25:41.991 | INFO    | Flow run 'fierce-jackrabbit' - ETL pipeline started

08:25:41.995 | INFO    | Task run 'extract-971' - Starting extract from /Users/earnch/projects/Data Engineer Technical Assignment/data/shopdata.db

08:25:42.027 | INFO    | Task run 'extract-971' - Loaded 12 rows from vw_raw_customers

08:25:42.032 | INFO    | Task run 'extract-971' - Loaded 20 rows from vw_raw_orders

08:25:42.034 | INFO    | Task run 'extract-971' - Loaded 15 rows from vw_exchange_rates

08:25:42.035 | INFO    | Task run 'extract-971' - Extract completed successfully

08:25:42.037 | INFO    | Task run 'extract-971' - Finished in state Completed()

08:25:42.997 | INFO    | Flow run 'fierce-jackrabbit' - Finished in state Completed()